<a href="https://colab.research.google.com/github/maggiecrowner/DS5001-Final-Project/blob/main/ParsedandAnnotatedData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parsed and Annotated Data

In [53]:
! git clone https://github.com/maggiecrowner/DS5001-Final-Project.git

fatal: destination path 'DS5001-Final-Project' already exists and is not an empty directory.


In [54]:
import pandas as pd
import glob
import os

path = '/content/DS5001-Final-Project/data/Source Data'
# read in only the first 18 csv files, to meet row requirements
csv_files = glob.glob(os.path.join(path, '**/*.csv'), recursive=True)[:18]

print(f"Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f"  {os.path.relpath(f, path)}")

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    df['_source_file'] = os.path.basename(f)
    dfs.append(df)

source_data = pd.concat(dfs, ignore_index=True)
print(f"\n Corpus shape: {source_data.shape}")
source_data.head()

Found 18 CSV files:
  LadyGaga.csv
  Khalid.csv
  Eminem.csv
  BillieEilish.csv
  CardiB.csv
  ArianaGrande.csv
  JustinBieber.csv
  NickiMinaj.csv
  SelenaGomez.csv
  PostMalone.csv
  Maroon5.csv
  TaylorSwift.csv
  Drake.csv
  DuaLipa.csv
  Beyonce.csv
  CharliePuth.csv
  KatyPerry.csv
  ColdPlay.csv

 Corpus shape: (5048, 8)


,Unnamed: 0,Artist,Title,Album,Year,Date,Lyric,_source_file
0,0.0,Lady Gaga,Do What U Want,ARTPOP,2013.0,2013-10-21,lady gaga r kelly yeah oh turn the mic up yea...,LadyGaga.csv
1,1.0,Lady Gaga,Always Remember Us This Way,A Star is Born (Original Motion Picture Soundt...,2018.0,2018-10-05,that arizona sky burnin' in your eyes you look...,LadyGaga.csv
2,2.0,Lady Gaga,Bad Romance,The Fame Monster,2009.0,2009-10-23,ohohohohoh ohohohoh ohohoh caught in a bad rom...,LadyGaga.csv
3,3.0,Lady Gaga,Sexxx Dreams,ARTPOP,2013.0,2013-11-06,last night our lovers' quarrel i was thinking ...,LadyGaga.csv
4,4.0,Lady Gaga,I’ll Never Love Again (Extended Version),A Star is Born (Original Motion Picture Soundt...,2018.0,2018-10-05,wish i could i could've said goodbye i would'v...,LadyGaga.csv


## LIB

In [55]:
source_data['track_id'] = source_data['Artist'] + ' — ' + source_data['Title'] + ' (' + source_data['Album'] + ')'

# add additional columns for model summarization
source_data['doc_length_words'] = source_data['Lyric'].astype(str).str.split().str.len()
source_data['doc_length_chars'] = source_data['Lyric'].astype(str).str.len()
source_data['Year'] = pd.to_numeric(source_data['Year'], errors='coerce')
source_data['Decade'] = (source_data['Year'].dropna().astype(int) // 10 * 10).astype(str) + 's'

LIB = (source_data[['track_id', 'Artist', 'Album', 'Title', 'Year', 'Decade',
                     'doc_length_words', 'doc_length_chars']]
       .drop_duplicates()
       .set_index('track_id'))

print(f" LIB shape: {LIB.shape}")
LIB.head()

 LIB shape: (5048, 7)


,Artist,Album,Title,Year,Decade,doc_length_words,doc_length_chars
track_id,,,,,,,
Lady Gaga — Do What U Want (ARTPOP),Lady Gaga,ARTPOP,Do What U Want,2013.0,2010s,607,2749
Lady Gaga — Always Remember Us This Way (A Star is Born (Original Motion Picture Soundtrack)),Lady Gaga,A Star is Born (Original Motion Picture Soundt...,Always Remember Us This Way,2018.0,2010s,234,1125
Lady Gaga — Bad Romance (The Fame Monster),Lady Gaga,The Fame Monster,Bad Romance,2009.0,2000s,533,2765
Lady Gaga — Sexxx Dreams (ARTPOP),Lady Gaga,ARTPOP,Sexxx Dreams,2013.0,2010s,430,2044
Lady Gaga — I’ll Never Love Again (Extended Version) (A Star is Born (Original Motion Picture Soundtrack)),Lady Gaga,A Star is Born (Original Motion Picture Soundt...,I’ll Never Love Again (Extended Version),2018.0,2010s,324,1648


In [56]:
LIB.to_csv('/content/DS5001-Final-Project/data/LIB.csv', sep='|', index=False)

## CORPUS

In [57]:
import nltk
import re
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt')
nltk.download('punkt_tab')

pos_group_map = {
    'NN': 'NOUN', 'NNS': 'NOUN', 'NNP': 'NOUN', 'NNPS': 'NOUN',
    'VB': 'VERB', 'VBD': 'VERB', 'VBG': 'VERB', 'VBN': 'VERB', 'VBP': 'VERB', 'VBZ': 'VERB',
    'JJ': 'ADJ',  'JJR': 'ADJ',  'JJS': 'ADJ',
    'RB': 'ADV',  'RBR': 'ADV',  'RBS': 'ADV',
}

records = []
for _, row in source_data.iterrows():
    artist = row['Artist']
    album  = row['Album']
    track  = row['Title']
    lyric  = str(row['Lyric'])

    words    = lyric.split()
    pos_tags = nltk.pos_tag(words)

    for word_id, (token_str, pos) in enumerate(pos_tags):
        term_str  = re.sub(r'[^\w\s]', '', token_str).lower()
        pos_group = pos_group_map.get(pos, 'OTHER')

        records.append({
            'Artist':    artist,
            'Album':     album,
            'Title':     track,
            'WordID':    word_id,
            'token_str': token_str,
            'term_str':  term_str,
            'pos':       pos,
            'pos_group': pos_group,
        })

CORPUS = pd.DataFrame(records)
CORPUS = CORPUS.set_index(['Artist', 'Album', 'Title', 'WordID'])
print(f" Corpus shape: {CORPUS.shape}")
CORPUS.head()

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


 Corpus shape: (1862580, 4)


token_str term_str pos pos_group
Artist    Album  Title          WordID                                 
Lady Gaga ARTPOP Do What U Want 0           lady     lady  NN      NOUN
                                1           gaga     gaga  NN      NOUN
                                2              r        r  NN      NOUN
                                3          kelly    kelly  RB       ADV
                                4           yeah     yeah  JJ       ADJ

In [58]:
CORPUS.to_csv('/content/DS5001-Final-Project/data/CORPUS.csv', sep='|', index=False)

## VOCAB

In [59]:
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import numpy as np

ps = PorterStemmer()

n = CORPUS['term_str'].value_counts().rename('n')
doc_term = CORPUS.reset_index().groupby('Title')['term_str'].apply(set)
N_docs = len(doc_term)

df_counts = (CORPUS.reset_index()
             .groupby('term_str')['Title']
             .nunique()
             .rename('df'))

VOCAB = pd.DataFrame({'n': n, 'df': df_counts})
VOCAB.index.name = 'term_str'

VOCAB['p']            = VOCAB['df'] / N_docs
VOCAB['i']            = np.log2(N_docs / VOCAB['df'])
VOCAB['dfidf']        = VOCAB['df'] * VOCAB['i']
VOCAB['porter_stem']  = VOCAB.index.map(lambda t: ps.stem(t))
VOCAB['max_pos']      = CORPUS.groupby('term_str')['pos'].agg(lambda x: x.value_counts().idxmax())
VOCAB['max_pos_group']= CORPUS.groupby('term_str')['pos_group'].agg(lambda x: x.value_counts().idxmax())
VOCAB['stop']         = VOCAB.index.map(lambda t: int(t in ENGLISH_STOP_WORDS))
VOCAB['ngram_length'] = VOCAB.index.map(lambda t: len(t.split()))
VOCAB = VOCAB.drop(columns=['df'])

print(f" VOCAB shape: {VOCAB.shape}")
VOCAB.head()

 VOCAB shape: (38061, 9)


,n,p,i,dfidf,porter_stem,max_pos,max_pos_group,stop,ngram_length
term_str,,,,,,,,,
,3,0.000612,10.673898,32.021694,,'',OTHER,0,0
0,328,0.038768,4.689005,890.910916,0,CD,OTHER,0,1
00,220,0.014691,6.088935,438.403351,00,CD,OTHER,0,1
000,77,0.003265,8.258860,132.141767,000,CD,OTHER,0,1
0000,1,0.000204,12.258860,12.258860,0000,CD,OTHER,0,1


In [60]:
VOCAB.to_csv('/content/DS5001-Final-Project/data/VOCAB.csv', sep='|')

# EDA / Summary statistics of the tables

In [61]:
print("Average length of document, in characters:", source_data['Lyric'].str.len().mean())

Average length of document, in characters: 1830.0605577689244


In [62]:
print("Number of observations/tokens:", len(CORPUS['token_str']))

Number of observations/tokens: 1862580


In [63]:
print("Number of observations/vocab:", len(VOCAB['dfidf']))

Number of observations/vocab: 38061


In [64]:
top20 = (VOCAB[VOCAB['stop'] == 0]
         .sort_values('dfidf', ascending=False)
         .head(20))
print("Top 20 significant words by DFIDF:")
top20[['n', 'p', 'i', 'dfidf', 'porter_stem', 'max_pos_group']]

Top 20 significant words by DFIDF:


,n,p,i,dfidf,porter_stem,max_pos_group
term_str,,,,,,
time,4857,0.375842,1.411803,2600.541283,time,NOUN
make,4507,0.352581,1.503973,2598.865222,make,VERB
say,5232,0.384003,1.380810,2598.683514,say,VERB
baby,7428,0.351561,1.508153,2598.548386,babi,NOUN
want,6728,0.341155,1.551501,2594.110172,want,VERB
youre,7413,0.401959,1.314881,2590.314619,your,NOUN
wanna,5530,0.325852,1.617712,2583.485799,wanna,VERB
let,5012,0.321975,1.634979,2579.996770,let,VERB
oh,10592,0.417058,1.261681,2578.875863,oh,OTHER
